In [1]:
"""
GRU 端到端 demo —— 云端推理入口（自包含 PyTorch，不引用外部模块）

本地：python gru_demo/gru_train.py -> 上传 gru_model.json + gru_train.py + 本 notebook
训练区间：2022-01-01 ~ 2023-09-28（含公榜评估窗口）
"""
import json
import os
import time

import dai
import numpy as np
import pandas as pd
import structlog
import torch
import torch.nn as nn

logger = structlog.get_logger()

MODEL_PATH = "gru_model.json"
INSTRUMENTS_TABLE = "bigalpha_2026_instruments"
OFFICIAL_EVAL_START = "2023-07-03"
OFFICIAL_EVAL_END = "2023-09-28"
BATCH = 512
FEATURE_PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
FEATURE_VOL_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
FEATURE_COLS = FEATURE_PRICE_COLS + FEATURE_VOL_COLS
VOL_COLS = FEATURE_VOL_COLS
N_FEAT = len(FEATURE_COLS)
WARMUP_TRADING_DAYS = 42


def warmup_start_date(start_date: str, n_trading_days: int = WARMUP_TRADING_DAYS) -> str:
    sd = pd.to_datetime(start_date)
    return (sd - pd.tseries.offsets.BDay(n_trading_days)).strftime("%Y-%m-%d")


def _ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        raise RuntimeError(f"缺少特征列: {missing}")
    return df


def preprocess_cloud(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    drop_suffixes = ("4", "5")
    drop_prefixes = (
        "ask_price", "bid_price",
        "ask_volume", "bid_volume",
        "ask_num_orders", "bid_num_orders",
    )
    drop_cols = [
        c for c in df.columns
        if any(c.startswith(p) and c[len(p):] in drop_suffixes for p in drop_prefixes)
    ]
    df = df.drop(columns=drop_cols, errors="ignore")
    if "instrument" not in df.columns:
        raise RuntimeError("云端 stock 表缺少 instrument 列")
    df["instrument"] = df["instrument"].astype(str)
    return _ensure_feature_columns(df)


def pool(start_date: str, end_date: str) -> list[str]:
    df = dai.query(
        f"SELECT DISTINCT instrument FROM {INSTRUMENTS_TABLE}",
        filters={"date": [start_date, end_date]},
    ).df()
    if df.empty:
        raise RuntimeError(f"{INSTRUMENTS_TABLE} 无成分股: {start_date}~{end_date}")
    return df["instrument"].astype(str).tolist()


class StockGRU(nn.Module):
    def __init__(self, n_feat, hidden=32, num_layers=1, seq_len=64):
        super().__init__()
        self.seq_len = seq_len
        self.gru = nn.GRU(n_feat, hidden, num_layers=num_layers, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, 1))

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def load_model(model_path=MODEL_PATH, map_location="cpu"):
    with open(model_path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    sd = {}
    for k, meta in payload["state_dict"].items():
        t = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
        sd[k] = t.reshape(meta["shape"]).to(map_location)
    ckpt = {k: v for k, v in payload.items() if k != "state_dict"}
    ckpt["state_dict"] = sd
    return ckpt


def build_dataset(table, sd, ed, mode, instruments, stats=None, seq_len=64):
    t0 = time.time()
    buf = warmup_start_date(sd)
    sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {table} ORDER BY instrument, date"
    df = dai.query(sql, filters={"date": [buf, ed]}).df()
    if df.empty:
        raise RuntimeError(f"{table} 无数据: {buf} ~ {ed}")
    df = preprocess_cloud(df)
    ins_set = set(instruments)
    df = df[df["instrument"].isin(ins_set)].copy()
    if df.empty:
        raise RuntimeError(f"过滤 instrument 后无数据: {buf} ~ {ed}")
    for c in VOL_COLS:
        df[c] = np.log1p(df[c].clip(lower=0))

    sd_ts, ed_ts = pd.to_datetime(sd), pd.to_datetime(ed)
    wins, ys, keys = [], [], []
    for ins, sub in df.groupby("instrument", sort=False):
        if len(sub) <= seq_len:
            continue
        feats = sub[FEATURE_COLS].to_numpy(np.float32)
        day = sub["date"].dt.normalize().to_numpy()
        close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
        close_px = sub["close"].to_numpy(np.float64)[close_pos]
        dates = day[close_pos]
        for k, p in enumerate(close_pos):
            d = pd.Timestamp(dates[k])
            if p + 1 < seq_len or d < sd_ts or d > ed_ts:
                continue
            label = None
            if k + 1 < len(close_pos) and close_px[k] > 0:
                r = close_px[k + 1] / close_px[k] - 1.0
                if np.isfinite(r):
                    label = np.float32(r)
            if mode == "train" and label is None:
                continue
            wins.append(feats[p - seq_len + 1: p + 1])
            ys.append(label if label is not None else np.float32(0.0))
            keys.append((d, ins))

    if not keys:
        raise RuntimeError(f"build_dataset 无样本 (mode={mode}, {sd}~{ed})")

    X = np.stack(wins).astype(np.float32)
    if stats is None:
        flat = X.reshape(-1, N_FEAT)
        stats = (flat.mean(0).astype(np.float32), flat.std(0).astype(np.float32) + 1e-6)
    m, s = stats
    X = ((X - m) / s).astype(np.float32)
    logger.info(f"{mode} 集构建完成", samples=len(keys), elapsed=round(time.time() - t0, 2))
    if mode == "train":
        return X, np.array(ys, np.float32), None, stats
    return X, None, pd.DataFrame(keys, columns=["date", "instrument"]), stats


def main(datasources, start_date, end_date, model_path=MODEL_PATH):
    table = datasources["bar1m"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"未找到模型文件 {model_path}，请先本地训练 gru_train.py 并上传 gru_model.json"
        )

    ckpt = load_model(model_path, map_location=device)
    stats = (np.asarray(ckpt["mean"], np.float32), np.asarray(ckpt["std"], np.float32))
    seq_len = ckpt.get("seq_len", ckpt["model_cfg"]["seq_len"])
    model = StockGRU(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logger.info("已加载模型", path=model_path, device=str(device))

    instruments = pool(start_date, end_date)
    logger.info(
        "构建测试集并预测",
        warmup_from=warmup_start_date(start_date),
        start=str(start_date),
        end=str(end_date),
        instruments=len(instruments),
    )
    Xte, _, idx_df, _ = build_dataset(
        table, start_date, end_date, "infer", instruments, stats, seq_len=seq_len
    )
    preds = []
    Xte_t = torch.from_numpy(Xte)
    with torch.no_grad():
        for i in range(0, len(idx_df), BATCH):
            xb = Xte_t[i:i + BATCH].to(device)
            preds.append(model(xb).cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)

    stk = dai.query(
        f"SELECT date, instrument FROM {datasources.get('instruments', INSTRUMENTS_TABLE)}",
        filters={"date": [start_date, end_date]},
    ).df()
    result = (
        pd.merge(idx_df, stk, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
        .reset_index(drop=True)
    )
    logger.info(
        "分数构建完成",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date, end_date = OFFICIAL_EVAL_START, OFFICIAL_EVAL_END
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    M.bigalpha_eval._latest(
        factor_data=score_data,
        start_date=start_date,
        end_date=end_date,
        show=True,
    )


[2026-07-08 14:24:11] [info     ] 已加载模型                          device=cpu path=gru_model.json
[2026-07-08 14:24:11] [info     ] 构建测试集并预测                       end=2023-09-28 instruments=1001 start=2023-07-03 warmup_from=2023-05-04


: 